# 03 — RFM Segmentation & Business Recommendations

**Step 2.2** — Online Retail Analytics Platform | Dev B

This notebook:
1. Computes **RFM scores** (Recency, Frequency, Monetary) per customer
2. Assigns each customer to a **named business segment**
3. Generates **rule-based recommendations** with estimated profit impact

> All functions live in `src/recommendation_engine.py` and are called
> directly by Streamlit (Dev C).


## Setup

In [1]:
import pandas as pd
import numpy as np
import sys, os
sys.path.append(os.path.abspath('..'))

from src.data_cleaning import clean_data
from src.recommendation_engine import compute_rfm, get_segment_summary, generate_recommendations

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)


## Load & Clean Data

In [2]:
import os
DATA_PATH = '../data/cleaned_retail_data.csv'
if not os.path.exists(DATA_PATH):
    print(f"File not found at {DATA_PATH}. Please run 01_data_exploration.ipynb first.")
else:
    print("Loading cleaned dataset...")
    df_clean = pd.read_csv(DATA_PATH, parse_dates=['InvoiceDate'])
    print(f"Cleaned shape: {df_clean.shape}")
    display(df_clean.head())

Loading cleaned dataset...


Cleaned shape: (534130, 10)


,Invoice,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,IsReturn,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,False,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,False,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,20.34


---
## 1. RFM Scores per Customer

Each customer receives three scores (1–4 scale, 4 = best):
| Score | Meaning |
|-------|---------|
| **R** (Recency) | 4 = bought very recently |
| **F** (Frequency) | 4 = buys very often |
| **M** (Monetary) | 4 = spends the most |

`RFM_Score = R + F + M` → max 12 (Champions), min 3 (Lost)


In [3]:
rfm = compute_rfm(df_clean)
print(f"Total customers scored: {len(rfm):,}")
rfm.head(15)


Total customers scored: 4,338


,CustomerID,Recency,Monetary,Frequency,R_Score,F_Score,M_Score,RFM_Score,Segment
0,17428.0,1,"17,256.85",28,4,4,4,12,Champions
1,14422.0,1,"4,263.64",6,4,4,4,12,Champions
2,16678.0,3,"3,109.99",12,4,4,4,12,Champions
3,13456.0,18,"1,766.72",6,4,4,4,12,Champions
4,16684.0,4,"66,653.56",28,4,4,4,12,Champions
5,17655.0,11,"2,638.94",7,4,4,4,12,Champions
6,12856.0,8,"2,170.78",6,4,4,4,12,Champions
7,15218.0,11,"5,756.89",11,4,4,4,12,Champions
8,15214.0,2,"1,661.44",8,4,4,4,12,Champions
9,16701.0,9,"5,398.30",13,4,4,4,12,Champions


---
## 2. Customer Segment Distribution

> **Business value:** Know where most of your customers sit today,
> and which segments need immediate action.


In [4]:
seg_summary = get_segment_summary(rfm)
seg_summary


,Segment,Customers,Avg_Recency_Days,Avg_Frequency,Avg_Revenue,Total_Revenue
1,Champions,1317,17.11,9.39,"4,926.89","6,488,713.17"
0,At Risk,449,106.22,4.68,"2,128.69","955,780.72"
6,Potential Loyalists,1204,166.32,1.31,595.43,"716,901.83"
2,High Spenders,137,23.81,1.80,"2,442.12","334,570.17"
7,Recent Customers,528,25.62,1.36,312.56,"165,034.20"
4,Loyal Customers,206,21.78,3.05,463.28,"95,434.93"
5,Needs Attention,197,148.77,3.01,418.44,"82,433.64"
3,Lost,300,267.83,1.00,161.19,"48,358.23"


In [5]:
# % of customers per segment
seg_summary['Customer_%'] = (seg_summary['Customers'] / seg_summary['Customers'].sum() * 100).round(1)
seg_summary['Revenue_%']  = (seg_summary['Total_Revenue'] / seg_summary['Total_Revenue'].sum() * 100).round(1)
seg_summary[['Segment', 'Customers', 'Customer_%', 'Total_Revenue', 'Revenue_%']]


,Segment,Customers,Customer_%,Total_Revenue,Revenue_%
1,Champions,1317,30.40,"6,488,713.17",73.00
0,At Risk,449,10.40,"955,780.72",10.80
6,Potential Loyalists,1204,27.80,"716,901.83",8.10
2,High Spenders,137,3.20,"334,570.17",3.80
7,Recent Customers,528,12.20,"165,034.20",1.90
4,Loyal Customers,206,4.70,"95,434.93",1.10
5,Needs Attention,197,4.50,"82,433.64",0.90
3,Lost,300,6.90,"48,358.23",0.50


### Segment Reference Guide

| Segment | Description | Recommended Action |
|---------|-------------|-------------------|
| **Champions** | High R+F+M — your best customers | Reward & retain |
| **Loyal Customers** | Buy often and recently | Upsell premium products |
| **High Spenders** | Big basket, moderate frequency | Increase visit frequency |
| **Recent Customers** | New or newly returned | Nurture with onboarding flow |
| **Potential Loyalists** | Promising, not committed yet | Second-purchase incentive |
| **Needs Attention** | Frequency dropping | Re-engage before they leave |
| **At Risk** | Were great, now silent | Win-back campaign |
| **Lost** | Long gone, low value | Low-cost reactivation only |


---
## 3. Drill Into Key Segments


In [6]:
# Champions detail
champions = rfm[rfm['Segment'] == 'Champions']
print(f"Champions: {len(champions)} customers")
champions.head(10)


Champions: 1317 customers


,CustomerID,Recency,Monetary,Frequency,R_Score,F_Score,M_Score,RFM_Score,Segment
0,17428.0,1,"17,256.85",28,4,4,4,12,Champions
1,14422.0,1,"4,263.64",6,4,4,4,12,Champions
2,16678.0,3,"3,109.99",12,4,4,4,12,Champions
3,13456.0,18,"1,766.72",6,4,4,4,12,Champions
4,16684.0,4,"66,653.56",28,4,4,4,12,Champions
5,17655.0,11,"2,638.94",7,4,4,4,12,Champions
6,12856.0,8,"2,170.78",6,4,4,4,12,Champions
7,15218.0,11,"5,756.89",11,4,4,4,12,Champions
8,15214.0,2,"1,661.44",8,4,4,4,12,Champions
9,16701.0,9,"5,398.30",13,4,4,4,12,Champions


In [7]:
# At-risk detail
at_risk = rfm[rfm['Segment'] == 'At Risk']
print(f"At-Risk customers: {len(at_risk)}")
at_risk.head(10)


At-Risk customers: 449


,CustomerID,Recency,Monetary,Frequency,R_Score,F_Score,M_Score,RFM_Score,Segment
878,13875.0,54,"1,786.79",5,2,4,4,10,At Risk
884,14101.0,74,"5,976.79",6,2,4,4,10,At Risk
887,16081.0,56,"2,799.42",5,2,4,4,10,At Risk
888,14107.0,52,"2,703.23",7,2,4,4,10,At Risk
892,16098.0,88,"2,005.63",7,2,4,4,10,At Risk
893,16209.0,89,"2,262.62",5,2,4,4,10,At Risk
901,15874.0,64,"4,405.88",7,2,4,4,10,At Risk
906,15674.0,73,"2,446.60",5,2,4,4,10,At Risk
913,17400.0,126,"2,097.84",5,2,4,4,10,At Risk
915,15827.0,73,"1,836.09",14,2,4,4,10,At Risk


In [8]:
# One-time buyers
one_timers = rfm[rfm['Frequency'] == 1]
print(f"One-time buyers: {len(one_timers)} ({len(one_timers)/len(rfm)*100:.1f}% of all customers)")


One-time buyers: 1493 (34.4% of all customers)


---
## 4. Business Recommendations

> Automatically generated rule-based recommendations, each with an estimated
> profit impact % if the recommended action is executed.


In [9]:
recommendations = generate_recommendations(df_clean)

print(f"Generated {len(recommendations)} recommendations\n")
print("=" * 70)
for i, rec in enumerate(recommendations, 1):
    print(f"[{i}] {rec['title']}")
    print(f"    Type     : {rec['type']}")
    print(f"    Segment  : {rec['segment']}")
    print(f"    Priority : {rec['priority']}")
    print(f"    Impact   : ~+{rec['impact_pct']}% profit improvement")
    print(f"    Action   : {rec['message']}")
    print()


Generated 7 recommendations

[1] Reward Your Top Customers
    Type     : Retention
    Segment  : Champions
    Priority : High
    Impact   : ~+3.6% profit improvement
    Action   : You have 1317 Champion customers who generate 73.0% of total revenue. Launch an exclusive loyalty programme (early access, free shipping, personalised discounts) to keep them engaged and prevent churn.

[2] Convert Potential Loyalists to Loyal Customers
    Type     : Upsell
    Segment  : Potential Loyalists
    Priority : Medium
    Impact   : ~+3.5% profit improvement
    Action   : 1204 customers have shown interest but haven't committed yet. Offer a 'second purchase' incentive (e.g. 10% off next order) or introduce a membership scheme to move them into the Loyal tier.

[3] Convert One-Time Buyers to Repeat Customers
    Type     : Cross-Sell
    Segment  : Recent Customers
    Priority : High
    Impact   : ~+3.4% profit improvement
    Action   : 34.4% of your customers (1493) have only bought once

In [10]:
# Export recommendations as a DataFrame
rec_df = pd.DataFrame(recommendations)
rec_df


,type,segment,title,message,impact_pct,priority
0,Retention,Champions,Reward Your Top Customers,You have 1317 Champion customers who generate ...,3.60,High
1,Upsell,Potential Loyalists,Convert Potential Loyalists to Loyal Customers,1204 customers have shown interest but haven't...,3.50,Medium
2,Cross-Sell,Recent Customers,Convert One-Time Buyers to Repeat Customers,34.4% of your customers (1493) have only bough...,3.40,High
3,Quality,All,Investigate High-Return Products,48 products have a return rate above 20%. The ...,2.50,Medium
4,Win-Back,At Risk,Re-engage At-Risk Customers,449 customers who used to be valuable are now ...,2.20,High
5,Marketing,All,Time Promotions for Maximum Impact,Sales peak at 12:00 and on Thursdays. Schedule...,2.00,Medium
6,Pricing / Catalogue,All,Review or Discontinue Low-Revenue Products,20 products generate less than £2 total revenu...,1.50,Low


---
## Summary

Results from running all cells above (4,338 segmented customers, total monetary value 8,887,226 GBP).

| Segment | # Customers | % Revenue | Action |
|---------|------------|-----------|--------|
| Champions | **1,317** (30.4%) | **73.0%** | Loyalty programme -- exclusive perks & early access |
| At Risk | **449** (10.4%) | **10.8%** | Win-back email -- targeted discount or re-engagement offer |
| Potential Loyalists | **1,204** (27.8%) | **8.1%** | Follow-up campaign -- second purchase incentive |
| High Spenders | **137** (3.2%) | **3.8%** | Premium service tier -- personal account manager |
| Loyal Customers | **206** (4.7%) | **1.1%** | Reward & upsell -- introduce higher-value product bundles |
| Recent Customers | **528** (12.2%) | **1.9%** | Onboarding sequence -- guide second purchase within 30 days |
| Needs Attention | **197** (4.5%) | **0.9%** | Re-engagement -- time-limited offer before they go dormant |
| Lost | **300** (6.9%) | **0.5%** | Final win-back -- heavy discount or sunset from active list |

---
*Next step -> Step 2.3: Visualisations & Report (`notebooks/04_visualisations.ipynb`)*
